In [0]:
CREATE CATALOG IF NOT EXISTS db_test_sonia;
CREATE SCHEMA IF NOT EXISTS db_test_sonia.sch_bronze;
CREATE SCHEMA IF NOT EXISTS db_test_sonia.sch_silver;


-- Bronze : structure brute, partitionnée par un timestamp unique
CREATE OR REPLACE TABLE db_test_sonia.sch_bronze.sales_transactions
PARTITIONED BY (ingestion_ts)
AS
SELECT
  s.*,
  CAST(current_timestamp() AS TIMESTAMP) AS ingestion_ts
FROM samples.bakehouse.sales_transactions s
WHERE 1=0;

-- Silver : structure enrichie avec traçabilité
CREATE OR REPLACE TABLE db_test_sonia.sch_silver.sales_transactions
AS
SELECT
  s.*,
  sha2(to_json(struct(s.transactionID)), 256) AS _key_hash,
  sha2(to_json(struct(
    s.customerID, s.franchiseID, s.dateTime, s.product,
    s.quantity, s.unitPrice, s.totalPrice, s.paymentMethod, s.cardNumber
  )), 256) AS _change_hash,
  CAST(current_timestamp() AS TIMESTAMP) AS ingestion_ts,
  CAST(NULL AS TIMESTAMP) AS valid_from,
  CAST(NULL AS TIMESTAMP) AS valid_to
FROM samples.bakehouse.sales_transactions s
WHERE 1=0;


-------------------créer la table de paramètres--------------

CREATE TABLE IF NOT EXISTS db_test_sonia.sch_silver.param_tables (
  table_name     STRING,
  key_columns    STRING,
  change_columns STRING
);

MERGE INTO db_test_sonia.sch_silver.param_tables AS t
USING (
  SELECT
    'sales_transactions' AS table_name,
    'transactionID'      AS key_columns,
    'customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber' AS change_columns
) AS s
ON t.table_name = s.table_name
WHEN MATCHED THEN UPDATE SET
  t.key_columns = s.key_columns,
  t.change_columns = s.change_columns
WHEN NOT MATCHED THEN INSERT *;